# Pruning — Learning both Weights and Connections for Efficient Neural Networks (2015)

_Papers · Efficiency & Compression_

**Delete the small-magnitude weights of a trained net, then retrain the survivors — shrink the model an order of magnitude with no accuracy loss.**

---

This notebook is a practice scaffold for the **Pruning — Learning both Weights and Connections for Efficient Neural Networks (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example: threshold = q * std, then mask by magnitude. ---
w   = torch.tensor([0.80, -0.05, 0.30, 0.02, -0.60, 0.10])
std = w.std(unbiased=True).item()
q   = 0.5
tau = q * std                                   # threshold = quality * std  (paper, Sec 4)
mask = (w.abs() > tau).int()
print("std =", round(std, 4), " threshold =", round(tau, 4))
print("mask =", mask.tolist(), " pruned =", (w * mask).tolist())
# std = 0.458  threshold = 0.229
# mask = [1, 0, 1, 0, 1, 0]  pruned = [0.8, 0.0, 0.3, 0.0, -0.6, 0.0]   -> 50% sparsity


# --- 1. A synthetic, learnable 10-class problem (no downloads needed). ---
D, K = 64, 10
g = torch.Generator().manual_seed(1)
centers = torch.randn(K, D, generator=g) * 2.5
def make(n, gen):
    y = torch.randint(0, K, (n,), generator=gen)
    X = centers[y] + torch.randn(n, D, generator=gen) * 1.0
    return X, y
Xtr, ytr = make(4000, g)
Xte, yte = make(1000, g)


# --- 2. A dense multi-layer perceptron (we import nn.Linear; the pruning is ours). ---
class MLP(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        self.fc1 = nn.Linear(D, h); self.fc2 = nn.Linear(h, h); self.fc3 = nn.Linear(h, K)
        self.act = nn.ReLU()
    def forward(self, x):
        x = self.act(self.fc1(x)); x = self.act(self.fc2(x)); return self.fc3(x)

def lin(net):                                   # the weight-bearing layers we prune
    return [net.fc1, net.fc2, net.fc3]

def acc(net, X, y):
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()

def train(net, epochs, lr, mask=None):
    opt = torch.optim.Adam(net.parameters(), lr=lr); lf = nn.CrossEntropyLoss(); net.train()
    for _ in range(epochs):
        opt.zero_grad(); lf(net(Xtr), ytr).backward(); opt.step()
        if mask is not None:                    # KEEP pruned weights at zero after every step
            with torch.no_grad():
                for i, l in enumerate(lin(net)):
                    l.weight.mul_(mask[i])
    return net


# --- 3. The novel part: magnitude pruning to a target sparsity (global threshold). ---
def make_mask(net, sparsity):
    all_w = torch.cat([l.weight.detach().abs().flatten() for l in lin(net)])
    k = int(sparsity * all_w.numel())
    tau = -1.0 if k == 0 else all_w.kthvalue(k).values.item()   # threshold = the k-th smallest size
    return {i: (l.weight.detach().abs() > tau).float() for i, l in enumerate(lin(net))}

def apply_mask(net, mask):
    with torch.no_grad():
        for i, l in enumerate(lin(net)):
            l.weight.mul_(mask[i])

def real_sparsity(net):
    tot = sum(l.weight.numel() for l in lin(net))
    zero = sum((l.weight.detach() == 0).sum().item() for l in lin(net))
    return zero / tot


# --- 4. Train dense baseline, then the ablation: prune-no-retrain vs prune-then-retrain. ---
base = MLP(); train(base, epochs=80, lr=0.005)
print("\nDENSE baseline test acc:", round(acc(base, Xte, yte), 4))

print("\n sparsity | no-retrain | retrain")
for s in [0.5, 0.7, 0.8, 0.9, 0.95]:
    a = copy.deepcopy(base); m = make_mask(a, s); apply_mask(a, m)
    no_rt = acc(a, Xte, yte)                                    # prune only
    b = copy.deepcopy(base); mb = make_mask(b, s); apply_mask(b, mb)
    train(b, epochs=50, lr=0.002, mask=mb)                      # prune THEN retrain survivors
    rt = acc(b, Xte, yte)
    print(f"   {real_sparsity(a):.2f}   |   {no_rt:.3f}    |  {rt:.3f}")

# Our small run (not the paper's numbers): up to ~80% sparsity both stay ~1.00 (redundant weights).
# At 90% the no-retrain net drops while retraining recovers it; at 95% retraining only partly recovers
# -- the limit of one-shot pruning (the paper ITERATES, Sec 3.4, to push sparsity higher).

## Visualize the data & results

_As we prune more weights of a trained net, what happens to test accuracy WITHOUT retraining vs WITH retraining the survivors?_

In [ ]:
import copy, torch
import torch.nn as nn
torch.manual_seed(0)

D, K = 64, 10
g = torch.Generator().manual_seed(1)
centers = torch.randn(K, D, generator=g) * 2.5
def make(n, gen):
    y = torch.randint(0, K, (n,), generator=gen)
    X = centers[y] + torch.randn(n, D, generator=gen) * 1.0
    return X, y
Xtr, ytr = make(4000, g); Xte, yte = make(1000, g)

class MLP(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        self.fc1 = nn.Linear(D, h); self.fc2 = nn.Linear(h, h); self.fc3 = nn.Linear(h, K)
        self.act = nn.ReLU()
    def forward(self, x):
        x = self.act(self.fc1(x)); x = self.act(self.fc2(x)); return self.fc3(x)

def lin(n): return [n.fc1, n.fc2, n.fc3]
def acc(n, X, y):
    n.eval()
    with torch.no_grad(): return (n(X).argmax(1) == y).float().mean().item()
def train(n, ep, lr, mask=None):
    opt = torch.optim.Adam(n.parameters(), lr=lr); lf = nn.CrossEntropyLoss(); n.train()
    for _ in range(ep):
        opt.zero_grad(); lf(n(Xtr), ytr).backward(); opt.step()
        if mask is not None:
            with torch.no_grad():
                for i, l in enumerate(lin(n)): l.weight.mul_(mask[i])
def make_mask(n, s):
    aw = torch.cat([l.weight.detach().abs().flatten() for l in lin(n)])
    k = int(s * aw.numel()); tau = -1.0 if k == 0 else aw.kthvalue(k).values.item()
    return {i: (l.weight.detach().abs() > tau).float() for i, l in enumerate(lin(n))}
def apply_mask(n, m):
    with torch.no_grad():
        for i, l in enumerate(lin(n)): l.weight.mul_(m[i])

base = MLP(); train(base, 80, 0.005)
no_rt, rt = [], []
for s in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99]:
    a = copy.deepcopy(base); m = make_mask(a, s); apply_mask(a, m); no_rt.append((s, round(acc(a, Xte, yte), 3)))
    b = copy.deepcopy(base); mb = make_mask(b, s); apply_mask(b, mb); train(b, 50, 0.002, mb); rt.append((s, round(acc(b, Xte, yte), 3)))
print("Prune, no retrain :", no_rt)
print("Prune, then retrain:", rt)
# Prune, no retrain : [(0.0,1.0),(0.5,1.0),(0.7,1.0),(0.8,1.0),(0.9,0.905),(0.95,0.184),(0.97,0.093),(0.99,0.093)]
# Prune, then retrain: [(0.0,1.0),(0.5,1.0),(0.7,1.0),(0.8,1.0),(0.9,1.0),(0.95,0.667),(0.97,0.093),(0.99,0.093)]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a dense net at high accuracy. You prune it to 90% sparsity (delete
            the 90% smallest-magnitude weights). You measure accuracy, then retrain only the survivors and
            measure again. What do you expect for each, and what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Prune to 90% sparsity and test without retraining: accuracy drops noticeably (in our run, from 100% to ~90%). — _Deleting weights changes every neuron's weighted sum a little; across the whole net those small errors accumulate._
- Keep the surviving weights, retrain a few steps re-applying the mask each step, and test again: accuracy climbs back up (in our run, back to ~100%). — _The survivors re-absorb the work the deleted connections did; starting from the trained weights preserves the co-adapted features (&sect;3.3)._
- Conclude that the recipe is prune and retrain, not prune alone. — _Pruning proposes which connections to drop; retraining pays the residual error. The paper calls the retrain step "critical."_

**Answer:** Without retraining, accuracy falls at 90% sparsity; with retraining it recovers to near the
                 dense baseline. Since the only difference is the retrain step, this isolates retraining as the
                 reason high sparsity is safe &mdash; exactly the paper's claim that the step is critical. Push
                 to 95%+ and even retraining cannot fully recover, marking the limit of one-shot pruning (the
                 paper iterates to go further, &sect;3.4).

</details>

**Problem 2.** A layer has weights $w = [0.80, -0.05, 0.30, 0.02, -0.60, 0.10]$ with standard deviation
            $\sigma \approx 0.458$. Using the paper's rule with quality $q = 0.5$, which weights are pruned,
            and what sparsity does that give?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the threshold $\tau = q\,\sigma = 0.5 \times 0.458 \approx 0.229$. — _The paper sets the threshold to a quality parameter times the layer's weight standard deviation (&sect;4)._
- Compare each $|w_i|$ to $\tau$: sizes are $[0.80, 0.05, 0.30, 0.02, 0.60, 0.10]$. — _Pruning scores by absolute value &mdash; sign does not matter, magnitude does._
- Keep $|w_i| \ge 0.229$: $\{0.80, 0.30, 0.60\}$ survive; $\{0.05, 0.02, 0.10\}$ are zeroed. — _Three of six weights pass the threshold._

**Answer:** Pruned weights: the $-0.05$, $0.02$, and $0.10$ (all below $\tau \approx 0.229$). The mask
                 is $[1,0,1,0,1,0]$, so the layer becomes $[0.80, 0, 0.30, 0, -0.60, 0]$ &mdash; 3 of 6 kept,
                 i.e. 50% sparsity. Note $-0.60$ survives: it is large in magnitude despite the negative
                 sign.

</details>

**Problem 3.** During retraining of a pruned net, you forget to re-apply the mask after each optimizer step. After
            a few epochs your network is no longer sparse. Explain what happened and how to fix it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the optimizer updates every parameter using its gradient, including the weights you zeroed. — _A zeroed weight still has a gradient; backprop will push it away from zero on the next step._
- Re-apply the mask immediately after opt.step() every iteration: W.mul_(mask). — _This forces the pruned weights back to exactly zero before the next forward pass, holding sparsity fixed._
- Verify sparsity afterward by counting zeros in each weight matrix. — _An honest sparse-retrain keeps the same set of weights at zero throughout; checking confirms the mask stuck._

**Answer:** The optimizer refilled the pruned weights: gradients are non-zero on zeroed connections, so
                 each step nudges them off zero and the holes close. Fix it by re-applying the mask after every
                 optimizer step (W.mul_(mask)), which pins the pruned weights at zero throughout
                 retraining. Then verify by counting zeros that the target sparsity is preserved.

</details>